In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import DataLoader
import lightning.pytorch as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.5.1+cu121
CUDA available: True


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATA_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet")

df = pd.read_parquet(DATA_PATH).copy()

# --- CONFIG ---
TIME_COL = "time"
PRICE_COL = "mid_c"
FEATURE_COLS = ["mid_o", "mid_h", "mid_l", "mid_c", "volume"]
# --------------

# Ensure datetime
df[TIME_COL] = pd.to_datetime(df[TIME_COL], utc=True)

# Sort by time
df = df.sort_values(TIME_COL).reset_index(drop=True)

# Series ID
df["series_id"] = "eurusd"

# Create time_idx in hours
df["time_idx"] = (df[TIME_COL] - df[TIME_COL].min()).dt.total_seconds() // 3600
df["time_idx"] = df["time_idx"].astype(int)

# Target = next hour's mid close
df["target"] = df[PRICE_COL].shift(-1)

# Remove final row (no target)
df = df.dropna(subset=["target"]).reset_index(drop=True)

print(df.head())
print(df.tail())
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

                       time  volume    mid_o    mid_h    mid_l    mid_c  \
0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   

     bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  \
0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787   
1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038   
2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159   
3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196   
4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245   

  series_id  time_idx   target  
0    eurusd         0  1.08029  
1    eurusd         

In [3]:
max_encoder_length = 48      # how many past hours the model sees
max_prediction_length = 1    # predict 1 hour ahead

# Train/validation split on time_idx
max_time_idx = df["time_idx"].max()
train_cutoff = int(max_time_idx * 0.8)

print("max_time_idx:", max_time_idx, "train_cutoff:", train_cutoff)

training = TimeSeriesDataSet(
    df[df["time_idx"] <= train_cutoff],
    time_idx="time_idx",
    target="target",
    group_ids=["series_id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_unknown_reals=FEATURE_COLS + ["target"],

    # 👉 IMPORTANT: allow gaps in time_idx (weekends, missing hours)
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df[df["time_idx"] > train_cutoff],
    predict=True,
    stop_randomization=True,
)

train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=64, num_workers=0)

print("Train batches:", len(train_dataloader))
print("Validation batches:", len(val_dataloader))


max_time_idx: 86564 train_cutoff: 69251
Train batches: 458
Validation batches: 1


In [4]:

pl.seed_everything(42)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    loss=QuantileLoss(),
    output_size=7, 
    log_interval=10,
    reduce_on_plateau_patience=4,
)

print("Model parameters (k):", tft.size()/1e3)


Seed set to 42


Model parameters (k): 59.057


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [5]:
trainer = pl.Trainer(
    max_epochs=15,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.1,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metri

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 458/458 [00:36<00:00, 12.43it/s, v_num=0, train_loss_step=0.000424, val_loss=0.000205, train_loss_epoch=0.000443]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 458/458 [00:36<00:00, 12.41it/s, v_num=0, train_loss_step=0.000424, val_loss=0.000205, train_loss_epoch=0.000443]


In [6]:
# get raw predictions
preds = tft.predict(val_dataloader, return_x=False)
preds = preds.detach().cpu().numpy().flatten()

# get true values
y_true = torch.cat([y[0] for x, y in iter(val_dataloader)]).detach().cpu().numpy().flatten()

# compare first 50 predictions
df_eval = pd.DataFrame({
    "true": y_true[:50],
    "pred": preds[:50],
})
df_eval

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


,true,pred
0,1.1513,1.151755


In [11]:
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

df_ind = df.copy()

# --- RSI(14) ---
df_ind["rsi"] = RSIIndicator(
    close=df_ind["mid_c"],
    window=14
).rsi()

# --- MACD (12,26,9) ---
macd = MACD(
    close=df_ind["mid_c"],
    window_slow=26,
    window_fast=12,
    window_sign=9
)
df_ind["macd"] = macd.macd()
df_ind["macd_signal"] = macd.macd_signal()

# --- Bollinger Bands (20, 2) ---
bb = BollingerBands(
    close=df_ind["mid_c"],
    window=20,
    window_dev=2
)
df_ind["bb_upper"] = bb.bollinger_hband()
df_ind["bb_middle"] = bb.bollinger_mavg()
df_ind["bb_lower"] = bb.bollinger_lband()

# Remove first rows where indicators have NaN
df_ind = df_ind.dropna().reset_index(drop=True)

df_ind.head(), df_ind.shape


(                       time  volume    mid_o    mid_h    mid_l    mid_c  \
 0 2016-01-08 09:00:00+00:00    1407  1.08722  1.08830  1.08670  1.08732   
 1 2016-01-08 10:00:00+00:00    1343  1.08734  1.08751  1.08585  1.08692   
 2 2016-01-08 11:00:00+00:00    1230  1.08694  1.08768  1.08612  1.08724   
 3 2016-01-08 12:00:00+00:00    1257  1.08724  1.08774  1.08590  1.08660   
 4 2016-01-08 13:00:00+00:00    7172  1.08658  1.08708  1.08032  1.08415   
 
      bid_o    bid_h    bid_l    bid_c  ...    ask_c  series_id  time_idx  \
 0  1.08715  1.08822  1.08662  1.08724  ...  1.08739     eurusd        33   
 1  1.08728  1.08744  1.08577  1.08685  ...  1.08699     eurusd        34   
 2  1.08687  1.08762  1.08605  1.08717  ...  1.08731     eurusd        35   
 3  1.08718  1.08766  1.08583  1.08653  ...  1.08668     eurusd        36   
 4  1.08650  1.08700  1.08021  1.08405  ...  1.08425     eurusd        37   
 
     target        rsi      macd  macd_signal  bb_upper  bb_middle  bb_lower  

In [9]:
pip install ta


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29497 sha256=a6ed5ae9f9d1ee6cc917f60788eb93e885587b9b398af11f06c7dfc1276c1fec
  Stored in directory: c:\users\admin\appdata\local\pip\cache\wheels\a1\d7\29\7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta
Note: you may need to restart the kernel to use updated packages.


In [12]:


FEATURE_COLS_BASE = ["mid_o", "mid_h", "mid_l", "mid_c", "volume"]
FEATURE_COLS_IND = [
    "rsi",
    "macd",
    "macd_signal",
    "bb_lower",
    "bb_middle",
    "bb_upper",
]
FEATURE_COLS_EXT = FEATURE_COLS_BASE + FEATURE_COLS_IND

max_encoder_length = 48
max_prediction_length = 1

max_time_idx_ind = df_ind["time_idx"].max()
train_cutoff_ind = int(max_time_idx_ind * 0.8)

print("max_time_idx_ind:", max_time_idx_ind, "train_cutoff_ind:", train_cutoff_ind)

training_ind = TimeSeriesDataSet(
    df_ind[df_ind["time_idx"] <= train_cutoff_ind],
    time_idx="time_idx",
    target="target",
    group_ids=["series_id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_unknown_reals=FEATURE_COLS_EXT + ["target"],

    allow_missing_timesteps=True,
)

validation_ind = TimeSeriesDataSet.from_dataset(
    training_ind,
    df_ind[df_ind["time_idx"] > train_cutoff_ind],
    predict=True,
    stop_randomization=True,
)

train_loader_ind = training_ind.to_dataloader(train=True, batch_size=64, num_workers=0)
val_loader_ind = validation_ind.to_dataloader(train=False, batch_size=64, num_workers=0)

print("Train batches (ind):", len(train_loader_ind))
print("Val batches (ind):", len(val_loader_ind))


max_time_idx_ind: 86564 train_cutoff_ind: 69251
Train batches (ind): 458
Val batches (ind): 1


In [13]:
pl.seed_everything(42)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1

tft_ind = TemporalFusionTransformer.from_dataset(
    training_ind,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    loss=QuantileLoss(),
    output_size=7,
    log_interval=10,
    reduce_on_plateau_patience=4,
)

print("Params (indicators model, k):", tft_ind.size()/1e3)

Seed set to 42


Params (indicators model, k): 65.867


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [16]:
from lightning.pytorch import Trainer

trainer_ind = Trainer(
    max_epochs=15,
    accelerator=accelerator,
    devices=devices,
    gradient_clip_val=0.1,
)

trainer_ind.fit(
    tft_ind,
    train_dataloaders=train_loader_ind,
    val_dataloaders=val_loader_ind,
)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 192    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 0      | train
5  | encoder_variable_selecti

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 458/458 [00:48<00:00,  9.50it/s, v_num=2, train_loss_step=0.000353, val_loss=0.00022, train_loss_epoch=0.000445] 

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 458/458 [00:48<00:00,  9.49it/s, v_num=2, train_loss_step=0.000353, val_loss=0.00022, train_loss_epoch=0.000445]


In [18]:
def predict_next_with_indicators(df_full,
                                 model,
                                 max_encoder_length=48,
                                 threshold_pips=2):
    """
    df_full: full dataframe with indicators (df_ind)
    model:   trained TFT model (e.g. tft_ind)
    threshold_pips: minimal predicted move (in pips) to act
    """
    # Work on a copy, sorted by time
    df_full = df_full.sort_values("time_idx").copy()

    # Keep just the last N rows to build a compact prediction dataset
    df_tail = df_full.tail(500).copy()  # 500 is arbitrary but > max_encoder_length

    # Build a fresh TimeSeriesDataSet for prediction
    pred_ds = TimeSeriesDataSet(
        df_tail,
        time_idx="time_idx",
        target="target",
        group_ids=["series_id"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=1,
        time_varying_unknown_reals=FEATURE_COLS_EXT + ["target"],
        allow_missing_timesteps=True,
    )

    pred_loader = pred_ds.to_dataloader(
        train=False,
        batch_size=1,
        num_workers=0,
    )

    # Get predictions for all sequences in df_tail
    preds = model.predict(pred_loader).detach().cpu().numpy().flatten()

    # The last prediction corresponds to the last time index in df_tail
    next_price_pred = float(preds[-1])

    # Last actual row = current state
    last_row = df_tail.iloc[-1]
    current_price = float(last_row["mid_c"])

    # indicators
    rsi = float(last_row["rsi"])
    macd = float(last_row["macd"])
    macd_signal = float(last_row["macd_signal"])
    bb_lower = float(last_row["bb_lower"])
    bb_middle = float(last_row["bb_middle"])
    bb_upper = float(last_row["bb_upper"])

    # pips threshold to price delta
    pip = 0.0001
    up_thresh = current_price + threshold_pips * pip
    down_thresh = current_price - threshold_pips * pip

    # basic model-based signal
    if next_price_pred > up_thresh:
        basic_signal = "BUY"
    elif next_price_pred < down_thresh:
        basic_signal = "SELL"
    else:
        basic_signal = "HOLD"

    # refine with indicators:
    #   - BUY only when momentum + not overheated
    #   - SELL only when downside momentum + not oversold
    final_signal = "HOLD"
    if basic_signal == "BUY" and (rsi < 70) and (macd > macd_signal):
        final_signal = "BUY"
    elif basic_signal == "SELL" and (rsi > 30) and (macd < macd_signal):
        final_signal = "SELL"

    return {
        "current_price": current_price,
        "predicted_next_price": next_price_pred,
        "rsi": rsi,
        "macd": macd,
        "macd_signal": macd_signal,
        "bb_lower": bb_lower,
        "bb_middle": bb_middle,
        "bb_upper": bb_upper,
        "basic_signal": basic_signal,
        "final_signal": final_signal,
    }

In [19]:
signal = predict_next_with_indicators(
    df_ind,
    tft_ind,
    max_encoder_length=48,
    threshold_pips=2,   # you can play with 1, 2, 3 pips, etc.
)
signal


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


{'current_price': 1.15176,
 'predicted_next_price': 1.1518077850341797,
 'rsi': 47.170324595485646,
 'macd': -0.0006269549680544273,
 'macd_signal': -0.0005473023028746182,
 'bb_lower': 1.1496065822591657,
 'bb_middle': 1.1523175,
 'bb_upper': 1.1550284177408345,
 'basic_signal': 'HOLD',
 'final_signal': 'HOLD'}

In [20]:
# Predictions on validation
y_pred = tft_ind.predict(val_loader_ind).detach().cpu().numpy().flatten()

# True values from dataloader
y_true = torch.cat([y[0] for x, y in iter(val_loader_ind)]).detach().cpu().numpy().flatten()

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

pip = 0.0001
mae_pips = mae / pip
rmse_pips = rmse / pip

print("MAE (price):", mae)
print("RMSE (price):", rmse)
print("MAE (pips):", mae_pips)
print("RMSE (pips):", rmse_pips)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


MAE (price): 0.0005007982
RMSE (price): 0.0005007982
MAE (pips): 5.00798225402832
RMSE (pips): 5.00798225402832
